# 21 — Seed Bagging: 10 modelos CatBoost (2 seeds x 5-fold)
## PRT Seguros

O bagging 5-fold one-hot (notebook `19`) foi o melhor resultado até agora (0.7383 no Kaggle).
Aqui dobramos o bagging: repetimos o 5-fold CV com uma segunda semente aleatória (outro
embaralhamento dos folds) e uma segunda semente de treino do CatBoost, totalizando **10 modelos**.
Mais modelos treinados em fatias/sementes diferentes → média mais estável → tende a generalizar
melhor ainda, sem mudar nada da lógica que já comprovamos que funciona.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

ID_COL, TARGET_COL = "cod_individuo", "churned"
MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

treino_completo = pd.concat([train, val], ignore_index=True)
feature_cols = [c for c in treino_completo.columns if c not in (ID_COL, TARGET_COL)]
X_full = treino_completo[feature_cols]
y_full = treino_completo[TARGET_COL]
X_kaggle = kaggle[feature_cols]

SEMENTES = [42, 123]
todas_kaggle_proba = []
todas_oof_auc = []

for semente in SEMENTES:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=semente)
    oof_proba = np.zeros(len(X_full))

    for fold, (idx_tr, idx_val_fold) in enumerate(skf.split(X_full, y_full)):
        X_tr_fold, y_tr_fold = X_full.iloc[idx_tr], y_full.iloc[idx_tr]
        X_val_fold, y_val_fold = X_full.iloc[idx_val_fold], y_full.iloc[idx_val_fold]

        idx_es_tr, idx_es_val = train_test_split(
            np.arange(len(X_tr_fold)), test_size=0.15, stratify=y_tr_fold, random_state=semente
        )
        neg_pos_ratio = (y_tr_fold.iloc[idx_es_tr] == 0).sum() / (y_tr_fold.iloc[idx_es_tr] == 1).sum()

        modelo = CatBoostClassifier(
            iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio,
            eval_metric="AUC", random_seed=semente, early_stopping_rounds=50, verbose=False,
        )
        modelo.fit(
            X_tr_fold.iloc[idx_es_tr], y_tr_fold.iloc[idx_es_tr],
            eval_set=(X_tr_fold.iloc[idx_es_val], y_tr_fold.iloc[idx_es_val]),
        )
        proba_fold = modelo.predict_proba(X_val_fold)[:, 1]
        oof_proba[idx_val_fold] = proba_fold
        todas_kaggle_proba.append(modelo.predict_proba(X_kaggle)[:, 1])
        print(f"seed {semente} fold {fold+1}/5 — AUC = {roc_auc_score(y_val_fold, proba_fold):.4f}")

    auc_seed = roc_auc_score(y_full, oof_proba)
    todas_oof_auc.append(auc_seed)
    print(f"-- seed {semente}: AUC-ROC OOF = {auc_seed:.4f}\n")

print(f"AUC-ROC OOF médio das 2 sementes: {np.mean(todas_oof_auc):.4f}")
print(f"Total de modelos no bagging: {len(todas_kaggle_proba)}")


seed 42 fold 1/5 — AUC = 0.8326


seed 42 fold 2/5 — AUC = 0.8260


seed 42 fold 3/5 — AUC = 0.8292


seed 42 fold 4/5 — AUC = 0.8244


seed 42 fold 5/5 — AUC = 0.8177
-- seed 42: AUC-ROC OOF = 0.8259



seed 123 fold 1/5 — AUC = 0.8259


seed 123 fold 2/5 — AUC = 0.8238


seed 123 fold 3/5 — AUC = 0.8195


seed 123 fold 4/5 — AUC = 0.8295


seed 123 fold 5/5 — AUC = 0.8318
-- seed 123: AUC-ROC OOF = 0.8259

AUC-ROC OOF médio das 2 sementes: 0.8259
Total de modelos no bagging: 10


In [2]:
proba_kaggle_final = np.mean(todas_kaggle_proba, axis=0)
submissao = pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle_final})
submissao.to_csv("submissions/submission_bagging_10seed.csv", index=False)
print("Salvo: submissions/submission_bagging_10seed.csv")


Salvo: submissions/submission_bagging_10seed.csv
